# ⚙️ The Optimizer Is Your Compiler

**Traditional compilation:**
```
Source Code  →  Compiler  →  Binary
```

**DSPy compilation:**
```
Signature + Metric + Data  →  Optimizer  →  Optimized Prompt
```

Both take **human-readable specifications** and produce **machine-executable artifacts**.

You never hand-write assembly. Why hand-write prompts?
The optimizer IS a compiler — it takes high-level intent and produces low-level instructions.

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, ipywidgets as widgets
from IPython.display import display
from dspy_tasks.tasks import get_task
from dspy_tasks.actions import run_baseline, run_optimization
from dspy_tasks.visualize import *

MODELS = ["github_copilot/gpt-4o", "github_copilot/gpt-4o-mini"]
model_dd = model_picker(MODELS)
opt_dd = optimizer_picker()
display(widgets.VBox([model_dd, opt_dd]))

## BootstrapFewShot: The Simple Compiler

**BootstrapFewShot** is the `-O1` of DSPy optimizers:
- It runs your module on training examples
- Keeps the ones that pass your metric
- Inserts the best examples as few-shot demonstrations in the prompt

Think of it like a compiler choosing which examples to **inline** — it picks the demonstrations
that best teach the model what you want.

In [ ]:
btn = run_button("Optimize Multi-hop QA")
out = widgets.Output()

def on_optimize(b):
    with out:
        out.clear_output()
        task = get_task("multihop_qa")
        print(f"⏳ Optimizing {task.name} with {opt_dd.value} on {model_dd.value}...")
        print(f"   This may take 10-60 seconds...\n")

        result = run_optimization("multihop_qa", model_dd.value, opt_dd.value, max_eval=8)

        display_improvement(result.baseline_score, result.optimized_score)
        print(f"⏱️  Optimization took {result.elapsed_seconds}s | {result.llm_calls} LLM calls")

        # THE KEY MOMENT: Show what changed
        display_prompt_diff(result.prompt_before, result.prompt_after)

        display_insight("What Just Happened?",
            f"DSPy tried different prompt configurations and found one that scores "
            f"{result.optimized_score:.0%} vs the baseline {result.baseline_score:.0%}. "
            "You didn't write a single prompt — the compiler did it for you.")

btn.on_click(on_optimize)
display(btn, out)

## MIPROv2: The Advanced Compiler

**MIPROv2** is the `-O3` of DSPy optimizers:
- Jointly optimizes **instructions** AND **examples** using Bayesian search
- Explores many candidate prompts, evaluating each against your metric
- More powerful but slower — like cranking up compiler optimization levels

When BootstrapFewShot isn't enough, MIPROv2 searches a larger space of possible prompts.

In [ ]:
compare_btn = run_button("Compare Optimizers")
compare_out = widgets.Output()

def on_compare_opt(b):
    with compare_out:
        compare_out.clear_output()
        task = get_task("ticket_routing")
        print(f"⏳ Comparing optimizers on {task.name}...\n")

        r_bs = run_optimization("ticket_routing", model_dd.value, "BootstrapFewShot", max_eval=8)
        print(f"BootstrapFewShot: {r_bs.baseline_score:.0%} → {r_bs.optimized_score:.0%} ({r_bs.elapsed_seconds}s)")

        r_mipro = run_optimization("ticket_routing", model_dd.value, "MIPROv2", max_eval=8)
        print(f"MIPROv2:          {r_mipro.baseline_score:.0%} → {r_mipro.optimized_score:.0%} ({r_mipro.elapsed_seconds}s)")

        scores = {
            "BootstrapFewShot": {"baseline": r_bs.baseline_score, "optimized": r_bs.optimized_score},
            "MIPROv2": {"baseline": r_mipro.baseline_score, "optimized": r_mipro.optimized_score},
        }
        fig = bar_comparison("Ticket Routing: Optimizer Comparison", scores)
        fig.show()

compare_btn.on_click(on_compare_opt)
display(compare_btn, compare_out)

In [ ]:
all_tasks = [(t.name, t.id) for t in [get_task(tid) for tid in ["multihop_qa", "ticket_routing", "report_generation"]]]
task_dd = widgets.Dropdown(options=all_tasks, description="Task:")
optimize_btn = run_button("Optimize & Show Diff")
optimize_out = widgets.Output()

def on_any_optimize(b):
    with optimize_out:
        optimize_out.clear_output()
        result = run_optimization(task_dd.value, model_dd.value, opt_dd.value, max_eval=8)
        display_improvement(result.baseline_score, result.optimized_score)
        display_prompt_diff(result.prompt_before, result.prompt_after)

optimize_btn.on_click(on_any_optimize)
display(widgets.HBox([task_dd, optimize_btn]), optimize_out)

## Key Takeaway

**The prompt is a compiled artifact.**

- When you change models → **recompile**
- When data changes → **recompile**
- When requirements change → **recompile**

The **metric + data = source code**. The optimizer compiles it into prompts.
Stop hand-tuning prompts. Write specifications and let the compiler do its job.